# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the `mlcroissant` library. The dataset contains mass spectrometry results on human extracellular vesicles, detailing protein abundance, modifications, and related metadata.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an mlcroissant.Metadata object (not dict/list)

print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Let's inspect all available record sets, the fields within each, and their `@id`s. In Croissant, you reference each entity using its `@id` to ensure clarity and reproducibility.

In [ ]:
# List available record sets and their fields using their @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s) in the dataset:\n")
for rs in record_sets:
    print(f"- Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (type: {field.data_type}, @id: {field.id})")
    print("")

## 3. Data Extraction
We'll load data from each record set into a DataFrame for analysis. You'll find all entities are referenced using their `@id` fields, ensuring unambiguous selection.

In [ ]:
# Extract data for each record set and store as DataFrames
dataframes = {}

for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))  # Reference record set using its @id
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded {len(df)} records from record set: {rs.name} (@id: {rs.id})\n")

# Example: show columns and sample records for the main record set
# Here, we use the first record set for further example (replace as needed)
main_rs = record_sets[0]
print(f"Columns in record set '{main_rs.name}' (@id: {main_rs.id}):")
print(dataframes[main_rs.id].columns.tolist())
dataframes[main_rs.id].head()

## 4. Exploratory Data Analysis (EDA)
Apply EDA: filter records, normalize numeric fields, and group by categorical fields.

First, let's show how to reference fields by their `@id`, then analyze one numeric field and one categorical field.

In [ ]:
# Find numeric and categorical fields by @id in the main record set
main_rs = record_sets[0]

# List field @ids for reference
print("Field @ids and types in the chosen record set:")
for field in main_rs.fields:
    print(f"- {field.name} (data_type: {field.data_type}, @id: {field.id})")

# Demo: select a field @id for numeric analysis
# Below are some example choices based on typical proteomic datasets:
# (Update @id if schema changes)
# Examples: cr:peptide_count, cr:MW, cr:Abundance
numeric_field_id = None
group_field_id = None
for field in main_rs.fields:
    # Use a field with numeric type
    if field.data_type in ('Number','Float','Integer','schema:Float','schema:Number','schema:Integer','http://schema.org/Float','http://schema.org/Number','http://schema.org/Integer') and numeric_field_id is None:
        numeric_field_id = field.id
    # Use a field with type Text (to group by)
    if field.data_type in ('Text','schema:Text','http://schema.org/Text') and group_field_id is None:
        group_field_id = field.id

# Display field @id selections
print(f"\nChosen numeric field @id: {numeric_field_id}")
print(f"Chosen group/categorical field @id: {group_field_id}")

# Proceed with filtering and normalization
df = dataframes[main_rs.id]
if numeric_field_id in df.columns:
    # Convert to numeric, coerce errors
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()  # Example: filter above mean
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())
    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records (first 5 rows):")
    print(filtered_df[[numeric_field_id, norm_col]].head())
    # Group by categorical field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped (mean) by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA. Please check field metadata.")

## 5. Visualization
Let's visualize the data distribution for the chosen numeric field and relationship to a group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group field exists
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        group_sizes = df[group_field_id].value_counts().nlargest(10).index
        sns.boxplot(
            x=group_field_id, y=numeric_field_id,
            data=df[df[group_field_id].isin(group_sizes)],
        )
        plt.title(f"{numeric_field_id} by {group_field_id} (top 10 groups)")
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("Could not visualize: Numeric field not found or not present in DataFrame.")

## 6. Conclusion
This notebook has demonstrated loading and exploring the FAIR² dataset using the `mlcroissant` library. We referenced all key data structures via Croissant `@id` fields, giving you a robust and reproducible workflow for further mass spectrometry data analysis.